In [ ]:
import numpy as np
import pandas as pd

from typing import Literal
from autogluon.tabular import TabularDataset, TabularPredictor
from pathlib import Path

In [ ]:
def build_df(type: Literal["train", "test"]) -> pd.DataFrame:
    log_df = pd.read_csv(f"./resources/kaggle/{type}_log.csv")

    data_dfs = [build_data_df(type, split) for split in sorted(log_df["split"].unique())]
    concat_data_df = pd.concat(data_dfs, ignore_index=True)

    df = log_df.merge(concat_data_df, on="object_id", how="left")

    df.drop(columns=["English Translation"], inplace=True)
    df.dropna(subset=["target"], inplace=True) if type == "train" else None

    return df

def compute_ts_features(group: pd.DataFrame) -> pd.Series:
    t = group["Time"].values
    f = group["Flux"].values

    feats = {}

    # --- Basic statistics ---
    feats["flux_mean"] = f.mean()
    feats["flux_std"] = f.std()
    feats["flux_min"] = f.min()
    feats["flux_max"] = f.max()
    feats["flux_median"] = np.median(f)
    feats["flux_iqr"] = np.percentile(f, 75) - np.percentile(f, 25)
    feats["n_obs"] = len(f)
    feats["flux_range"] = feats["flux_max"] - feats["flux_min"]

    # --- Time span ---
    feats["time_span"] = t.max() - t.min() if len(t) > 1 else 0.0

    # --- Trend (slope vs time) ---
    if len(f) > 1:
        slope, intercept = np.polyfit(t, f, 1)
        feats["flux_slope"] = slope
    else:
        feats["flux_slope"] = 0.0

    # --- First/last values ---
    feats["flux_first"] = f[0]
    feats["flux_last"] = f[-1]
    feats["flux_diff_last_first"] = f[-1] - f[0]

    # --- Mean absolute derivative ---
    if len(f) > 1:
        feats["flux_deriv_mean_abs"] = np.mean(np.abs(np.diff(f) / np.diff(t)))
    else:
        feats["flux_deriv_mean_abs"] = 0.0

    return pd.Series(feats)


def build_data_df(type: Literal["train", "test"], split: str) -> pd.DataFrame:
    # Load the lightcurve file for this split
    data_df = pd.read_csv(f"./resources/kaggle/{split}/{type}_full_lightcurves.csv")

    # Compute time-series features per (object_id, Filter)
    agg_data_df = (
        data_df
        .groupby(["object_id", "Filter"])
        .apply(compute_ts_features)
        .reset_index()
    )

    # Pivot: each filter becomes its own feature group
    pivot_data_df = agg_data_df.pivot_table(
        index="object_id",
        columns="Filter",
        aggfunc="first"
    )

    # Flatten multi-index columns into names like: flux_mean_g
    pivot_data_df.columns = [
        f"{stat}_{flt}" for stat, flt in pivot_data_df.columns
    ]

    return pivot_data_df.reset_index()


# def build_data_df(type: Literal["train", "test"], split: str) -> pd.DataFrame:
#     data_df = pd.read_csv(f"./resources/kaggle/{split}/{type}_full_lightcurves.csv")

#     agg_data_df = data_df.groupby(["object_id", "Filter"]).agg(
#         flux_mean=("Flux", "mean"),
#         flux_std=("Flux", "std"),
#         flux_min=("Flux", "min"),
#         flux_max=("Flux", "max"),
#         n_obs=("Flux", "count"),
#     ).reset_index()

#     pivot_data_df = agg_data_df.pivot(index="object_id", columns="Filter")
#     pivot_data_df.columns = [f"{stat}_{filter}" for stat, filter in pivot_data_df.columns]
#     pivot_data_df = pivot_data_df.reset_index()

#     return pivot_data_df

In [ ]:
train_df = build_df(type="train")
test_df = build_df(type="test")

predictor = TabularPredictor(path = "../AutogluonModels", problem_type="binary", label="target").fit(train_df, presets = "medium")

prediction_df = pd.DataFrame({
    "object_id": test_df["object_id"],
    "prediction": predictor.predict(test_df),
})

prediction_df

In [ ]:
Path("../output").mkdir(parents=True, exist_ok=True)

prediction_df.to_csv("../output/submission.csv", index=False)